In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000") # send everything to the local mlflow server
mlflow.set_experiment("db-delay-first-experiment") # everything will be logged under this experiment name
print(mlflow.__version__)

3.17.2


In [2]:
import polars as pl

from db_delay_mlops.paths import DATA_DIR

data_dir = DATA_DIR / "processed" / "first_experiment"
train = pl.read_parquet(data_dir / "train.parquet")
valid = pl.read_parquet(data_dir / "valid.parquet")

train.head()

id,eva,arrival_planned_time,stada_name,line,final_destination_station,train_group,train_line_station_num,minutes_since_departure,hour,weekday,month,temperature_2m,precipitation,weather_code,is_holiday,y
str,str,datetime[ns],str,str,str,str,i32,i64,i8,i8,i8,f64,f64,i16,bool,i8
"""8357206333431226217-2406302323…","""08000073""",2024-06-30 23:56:00,"""Singen (Hohentwiel)""","""S6""","""Engen""","""s_bahn""",12,33,23,7,6,15.4,0.0,3,false,0
"""6280290468004586530-2406302310…","""08000124""",2024-07-01 00:01:00,"""Gießen""","""RE99""","""Gießen""","""regional""",7,51,0,1,7,16.6,0.0,3,false,0
"""428488854693535724-2406302201-…","""08000134""",2024-07-01 00:02:00,"""Trier Hbf""","""RB71""","""Trier Hbf""","""regional""",35,121,0,1,7,16.7,0.6,61,false,0
"""-5012084765708061335-240630233…","""08002041""",2024-07-01 00:00:00,"""Frankfurt (Main) Süd""","""RB58""","""Hanau Hbf""","""regional""",4,27,0,1,7,16.5,0.3,61,false,0
"""2954494231059414669-2406302333…","""08000118""",2024-07-01 00:00:00,"""Gelsenkirchen Hbf""","""RB32""","""Duisburg Hbf""","""regional""",6,27,0,1,7,16.1,0.8,3,false,0


In [3]:
import numpy as np
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score


def evaluate(y_true: np.ndarray, p: np.ndarray) -> dict[str, float]:
    """All metrics we compare models on. p = predicted probability of being late."""
    return {
        "pr_auc": average_precision_score(y_true, p),
        "roc_auc": roc_auc_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
        "log_loss": log_loss(y_true, p),
    }


y_valid = valid["y"].to_numpy()

## Baselines: the yardstick for every later model

Every model follows the same three steps: **learn on `train`**, **predict on `valid`**, **evaluate**. The test set is not touched until the very end.

| Baseline | What it learns from `train` | How it predicts | What it shows |
| --- | --- | --- | --- |
| 1 · constant | 1 number: the share of late arrivals | the same probability for every stop | the score of a "model" that cannot tell trains apart |
| 2 · train group × hour | 72 numbers: late share per (group, hour) | looks up the stop's group and hour | what a simple pivot table already achieves |
| 3 · logistic regression | one weight per feature value | weighted sum of all features → probability | what the simplest real ML model achieves with all 13 features |

Every run gets the tag `stage = baseline`, so we can find and compare them later.

In [4]:
# Baseline 1: the same probability for every stop
with mlflow.start_run(run_name="baseline-constant"):
    p_train = train["y"].mean()  # share of late arrivals in train, learned from train only
    p_valid = np.full(len(y_valid), p_train)  # every stop in valid gets this value

    mlflow.set_tag("stage", "baseline")
    mlflow.log_params({"model": "constant", "p": round(p_train, 4), "n_train": train.height, "n_valid": valid.height})
    mlflow.log_metrics(evaluate(y_valid, p_valid))

print(f"p = {p_train:.4f}", evaluate(y_valid, p_valid))

🏃 View run baseline-constant at: http://127.0.0.1:5000/#/experiments/1/runs/bb668abfe3b74f4f84fe31de8491c17c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
p = 0.2090 {'pr_auc': 0.1972629425119825, 'roc_auc': 0.5, 'brier': 0.15848875119968517, 'log_loss': 0.49700930360186396}


In [5]:
# Baseline 2: late share per train group and hour (a lookup table with 3 x 24 = 72 rows)
with mlflow.start_run(run_name="baseline-group-hour"):
    rates = train.group_by("train_group", "hour").agg(p=pl.col("y").mean(), n=pl.len())  # learned from train only

    p_valid = (
        valid.join(rates, on=["train_group", "hour"], how="left")  # look up each stop
        .with_columns(pl.col("p").fill_null(p_train))  # combination not seen in train -> overall share
        ["p"]
        .to_numpy()
    )

    mlflow.set_tag("stage", "baseline")
    mlflow.log_params({"model": "rate_per_train_group_and_hour", "n_cells": rates.height, "n_train": train.height})
    mlflow.log_metrics(evaluate(y_valid, p_valid))

print(evaluate(y_valid, p_valid))
rates.sort("train_group", "hour").head(10)

🏃 View run baseline-group-hour at: http://127.0.0.1:5000/#/experiments/1/runs/2f8b0cd4759f4dc68c06e03277db6de4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
{'pr_auc': 0.35570705011697556, 'roc_auc': 0.6925697485390332, 'brier': 0.14515655116927004, 'log_loss': 0.45752605008541364}


train_group,hour,p,n
str,i8,f64,u32
"""long_distance""",0,0.561391,2818
"""long_distance""",1,0.50156,1603
"""long_distance""",2,0.479032,1240
"""long_distance""",3,0.430497,1187
"""long_distance""",4,0.347013,1657
"""long_distance""",5,0.271922,3387
"""long_distance""",6,0.242343,5191
"""long_distance""",7,0.245146,7004
"""long_distance""",8,0.287431,7988


### Baseline 3: logistic regression

A linear model computes a **weighted sum** of the features and turns it into a probability with the sigmoid function, $p = 1 / (1 + e^{-(w_0 + w_1 x_1 + w_2 x_2 + \dots)})$. It only understands numbers, so the data must be prepared first:

| Step | Applied to | Why |
| --- | --- | --- |
| Fill gaps with `"missing"` | text columns | `line` is empty for about 97 % of long-distance trains |
| **One-hot encoding** | station, line, destination, train group | "Köln Hbf" is not a number. Each value gets its own 0/1 column. Values with fewer than 100 rows are grouped into one "infrequent" column. |
| **One-hot encoding** | `hour`, `weekday`, `month`, `weather_code` | A linear model can only learn "the later, the more" for a number. But the late share peaks at 17:00 and falls again; weather code 95 is not "more" than 61. As categories, every hour gets its own weight. |
| Fill gaps with the median, **scaling** | other numbers | Minutes since departure (0–1,500) and precipitation (0–20 mm) get the same scale, so the optimizer converges |

Everything is wrapped in a **Pipeline**: the preparation is learned on `train` and saved together with the model, so the API later prepares new data exactly the same way.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TEXT_FEATURES = ["stada_name", "line", "final_destination_station", "train_group"]
CODE_FEATURES = ["hour", "weekday", "month", "weather_code"]  # numbers that are really categories
NUMERIC_FEATURES = ["train_line_station_num", "minutes_since_departure", "temperature_2m", "precipitation", "is_holiday"]
FEATURES = TEXT_FEATURES + CODE_FEATURES + NUMERIC_FEATURES


def to_model_input(df: pl.DataFrame):
    """Only the 13 features, missing text as "missing", as pandas for scikit-learn."""
    return df.select(FEATURES).with_columns(pl.col(TEXT_FEATURES).fill_null("missing")).to_pandas()


X_train, y_train = to_model_input(train), train["y"].to_numpy()
X_valid = to_model_input(valid)
print(X_train.shape, X_valid.shape)
X_train.head()

(1120128, 13) (285416, 13)


,stada_name,line,final_destination_station,train_group,hour,weekday,month,weather_code,train_line_station_num,minutes_since_departure,temperature_2m,precipitation,is_holiday
0,Singen (Hohentwiel),S6,Engen,s_bahn,23,7,6,3,12,33,15.4,0.0,False
1,Gießen,RE99,Gießen,regional,0,1,7,3,7,51,16.6,0.0,False
2,Trier Hbf,RB71,Trier Hbf,regional,0,1,7,61,35,121,16.7,0.6,False
3,Frankfurt (Main) Süd,RB58,Hanau Hbf,regional,0,1,7,61,4,27,16.5,0.3,False
4,Gelsenkirchen Hbf,RB32,Duisburg Hbf,regional,0,1,7,3,6,27,16.1,0.8,False


In [7]:
import time

import mlflow.sklearn

preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100), TEXT_FEATURES + CODE_FEATURES),
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC_FEATURES),
])
logreg = Pipeline([("preprocess", preprocess), ("model", LogisticRegression(max_iter=1000))])

with mlflow.start_run(run_name="baseline-logistic-regression"):
    start = time.perf_counter()
    logreg.fit(X_train, y_train)  # learns the one-hot columns, the scaling and the weights on train
    fit_seconds = time.perf_counter() - start

    p_valid = logreg.predict_proba(X_valid)[:, 1]  # column 1 = probability of y = 1 (late)

    mlflow.set_tag("stage", "baseline")
    mlflow.log_params({
        "model": "logistic_regression",
        "features": ",".join(FEATURES),
        "one_hot_min_frequency": 100,
        "n_one_hot_columns": logreg.named_steps["preprocess"].transform(X_train.head(1)).shape[1],
        "n_train": train.height,
    })
    mlflow.log_metrics(evaluate(y_valid, p_valid) | {"fit_seconds": fit_seconds})
    # Save the whole pipeline (preparation + model), with an example input for the signature
    mlflow.sklearn.log_model(
    logreg,
    name="model",
    input_example=X_valid.head(5),
    skops_trusted_types=["numpy.dtype"],  # describes a number type, executes no code
)


print(f"trained in {fit_seconds:.0f} s", evaluate(y_valid, p_valid))

/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run baseline-logistic-regression at: http://127.0.0.1:5000/#/experiments/1/runs/ded68ede10bb49d09f260032437cc392
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
trained in 31 s {'pr_auc': 0.4522550751984122, 'roc_auc': 0.773209478750801, 'brier': 0.13327172107714785, 'log_loss': 0.4180306169870499}


## Comparing the baselines

`mlflow.search_runs` returns the runs of the current experiment as a table. The same comparison is in the MLflow UI: tick the runs → **Compare**.

In [8]:
runs = mlflow.search_runs(filter_string="tags.stage = 'baseline'", order_by=["metrics.pr_auc DESC"])
runs[["tags.mlflow.runName", "metrics.pr_auc", "metrics.roc_auc", "metrics.brier", "metrics.log_loss", "metrics.fit_seconds"]].round(4)

,tags.mlflow.runName,metrics.pr_auc,metrics.roc_auc,metrics.brier,metrics.log_loss,metrics.fit_seconds
0,baseline-logistic-regression,0.4523,0.7732,0.1333,0.4180,31.3418
1,baseline-group-hour,0.3557,0.6926,0.1452,0.4575,NaN
2,baseline-constant,0.1973,0.5000,0.1585,0.4970,NaN


## Questions

1. The `roc_auc` of baseline 1 is exactly 0.5 and its `pr_auc` is about the late share of `valid`. Why?
2. Why are `p_train` and `rates` computed from `train` only, and not from `train` and `valid` together?
3. How much better is the logistic regression than baseline 2? Which of the 13 features could it use that baseline 2 does not have?
4. If you ran baseline 3 once more, would the result be exactly the same? Why (not)?

## Candidates: trying to beat the best baseline

The bar is the logistic regression: **PR-AUC 0.452**, Brier 0.133 on `valid`.

We compare three gradient boosting libraries with their **default settings**: LightGBM, XGBoost and CatBoost.
Trees can learn interactions (e.g. "S1 *in Munich*", "heat *in the afternoon*") that a linear model cannot.

Rules for a fair comparison:
- same `train` / `valid`, same 13 features, same `evaluate()`
- `test` is not used
- every run gets the tag `stage = candidate` and logs `fit_seconds`

The best one or two are tuned afterwards.

In [9]:
import mlflow.catboost
import mlflow.lightgbm
import mlflow.xgboost

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

### Preparation for tree models

Unlike the logistic regression, tree models need **no scaling and no one-hot encoding**:
a tree only asks "is the value above or below X?", so the scale does not matter,
and all three libraries handle categories themselves.

Two things are still needed:
- **LightGBM, XGBoost:** text columns become pandas categories with the categories of `train` (`CategoryCaster`)
- **CatBoost:** missing text becomes `"missing"` (`FillMissingText`), because CatBoost cannot handle `None` in categories

In [10]:
import time

import mlflow.sklearn
from sklearn.pipeline import Pipeline

from db_delay_mlops.models.preprocessing import CategoryCaster, FillMissingText

# Raw features: every pipeline prepares them itself (CategoryCaster, FillMissingText, ...).
# Unique names, so they cannot be mixed up with X_train / X_valid of the logistic regression.
X_train_raw = train.select(FEATURES).to_pandas()
X_valid_raw = valid.select(FEATURES).to_pandas()

def train_and_log(run_name: str, pipeline: Pipeline, stage: str, trusted_types: list[str]) -> dict[str, float]:
    """Fit the pipeline on train, evaluate it on valid and log everything to MLflow as one run.

    Uses the notebook variables X_train_raw, y_train, X_valid_raw, y_valid and FEATURES.
    Returns the metrics, so they can be printed or collected in a table.
    """
    model = pipeline[-1]  # the last step of the pipeline is the model itself

    with mlflow.start_run(run_name=run_name):
        start = time.perf_counter()
        pipeline.fit(X_train_raw, y_train)  # every step learns on train only
        fit_seconds = time.perf_counter() - start

        p_valid = pipeline.predict_proba(X_valid_raw)[:, 1]  # column 1 = probability of y = 1 (late)
        metrics = evaluate(y_valid, p_valid) | {"fit_seconds": fit_seconds}

        mlflow.set_tag("stage", stage)
        mlflow.log_params({
            "model": type(model).__name__,  # e.g. "LGBMClassifier"
            "features": ",".join(FEATURES),
            "n_train": len(X_train_raw),
        })
        mlflow.log_params(model.get_params())  # all settings of the model, for the record
        mlflow.log_metrics(metrics)
        # Save the whole pipeline (preparation + model), with an example input for the signature
        mlflow.sklearn.log_model(
            pipeline,
            name="model",
            input_example=X_valid_raw.head(5),
            skops_trusted_types=trusted_types,
        )

    print(f"{run_name}: {fit_seconds:.0f} s, pr_auc={metrics['pr_auc']:.4f}, brier={metrics['brier']:.4f}")
    return metrics

In [11]:
import pandas as pd
N_TREES = 100  # same number of trees for every tree model, so the comparison is fair

results = {}

results["lightgbm"] = train_and_log(
    run_name="lightgbm",
    pipeline=Pipeline([
        ("categories", CategoryCaster(columns=TEXT_FEATURES)),  # text -> category (learned on train)
        ("model", LGBMClassifier(n_estimators=N_TREES, random_state=42, verbose=-1)),
    ]),
    stage="candidate",
    trusted_types=[
        "collections.OrderedDict",
        "db_delay_mlops.models.preprocessing.CategoryCaster",
        "lightgbm.basic.Booster",
        "lightgbm.sklearn.LGBMClassifier",
    ],
)

results["xgboost"] = train_and_log(
    run_name="xgboost",
    pipeline=Pipeline([
        ("categories", CategoryCaster(columns=TEXT_FEATURES)),
        ("model", XGBClassifier(n_estimators=N_TREES, enable_categorical=True, tree_method="hist", random_state=42)),
    ]),
    stage="candidate",
    trusted_types=[
        "db_delay_mlops.models.preprocessing.CategoryCaster",
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBClassifier",
    ],
)

results["catboost"] = train_and_log(
    run_name="catboost",
    pipeline=Pipeline([
        ("fill_text", FillMissingText(columns=TEXT_FEATURES)),  # None -> "missing"
        ("model", CatBoostClassifier(
            n_estimators=N_TREES,
            cat_features=TEXT_FEATURES,  # these columns are categories, by name
            random_seed=42,
            verbose=0,
            thread_count=-1,  # use all CPU cores
            allow_writing_files=False,  # no catboost_info/ folder in notebooks/
        )),
    ]),
    stage="candidate",
    trusted_types=[
        "catboost.core.CatBoostClassifier",
        "db_delay_mlops.models.preprocessing.FillMissingText",
    ],
)

pd.DataFrame(results).T.round(4)  # one row per model

/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run lightgbm at: http://127.0.0.1:5000/#/experiments/1/runs/6dd7fd011624476b8c7ef5e534df9b22
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
lightgbm: 5 s, pr_auc=0.4942, brier=0.1283


/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run xgboost at: http://127.0.0.1:5000/#/experiments/1/runs/bbd0a342e7954134a1617b8c9a93f766
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
xgboost: 13 s, pr_auc=0.4924, brier=0.1292


/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run catboost at: http://127.0.0.1:5000/#/experiments/1/runs/16dbf7f55447470c90188cc9642a8908
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
catboost: 22 s, pr_auc=0.4695, brier=0.1318


,pr_auc,roc_auc,brier,log_loss,fit_seconds
lightgbm,0.4942,0.7923,0.1283,0.4041,5.1225
xgboost,0.4924,0.7901,0.1292,0.4084,12.8492
catboost,0.4695,0.7773,0.1318,0.4142,21.7689


In [14]:
runs = mlflow.search_runs(
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.pr_auc DESC"],
)
runs = runs[runs["tags.stage"].isin(["baseline", "candidate"])]
runs[["tags.mlflow.runName", "metrics.pr_auc", "metrics.roc_auc", "metrics.brier", "metrics.log_loss", "metrics.fit_seconds"]].round(4)

,tags.mlflow.runName,metrics.pr_auc,metrics.roc_auc,metrics.brier,metrics.log_loss,metrics.fit_seconds
0,lightgbm,0.4942,0.7923,0.1283,0.4041,5.1225
1,xgboost,0.4924,0.7901,0.1292,0.4084,12.8492
2,catboost,0.4695,0.7773,0.1318,0.4142,21.7689
3,baseline-logistic-regression,0.4523,0.7732,0.1333,0.4180,31.3418
4,baseline-group-hour,0.3557,0.6926,0.1452,0.4575,NaN
5,baseline-constant,0.1973,0.5000,0.1585,0.4970,NaN
